# Suite DOM — Value domains

A literal carries its value domain: a standard's representation (`Integer`, `Ieee754`, `Bits`, `Bytes`, `Unicode`,
`Ieee1164`, `Enum`, `Packed`, `Bool`), or by default its native's. Domains are data, compared by structure and
validated; a registered domain is written by name, any other by value, and a default one not at all. A value of another domain is a
typed value: compared within its domain, with integer and IEEE 754 arithmetic, the bitwise operations and the
conversions by domain; an operation's domain is its result's.

In [ ]:
import decimal
import math
import struct

from mbse.Expressions import Domains as D, Evaluators as V, Expressions as E
from mbse.Expressions import Translators as T
from mbse.Expressions.Dialects.Basic import Ieee754
from mbse.Expressions.Dialects.Python import Expressions as P
from mbse.Expressions.Framework import Domains as FD, Evaluators as FE, Terms as F
from mbse.Schemas.Framework import JSON, YAML, Comparison as C, Plain, Validators
from support import raises

I8 = D.OfInteger.Builder().width(8).create()
U8 = D.OfInteger.Builder().width(8).signed(False).overflow("wrap").create()
F32 = D.OfIeee754.Builder().format("binary32").create()
F16, F128, D64, D128 = (D.OfIeee754.Data(f) for f in ("binary16", "binary128", "decimal64", "decimal128"))
Bits2 = D.OfBits.Builder().width(2).create()
State = D.OfEnum.Builder().members("IDLE", "RUN").create()
Coded = D.OfPacked.Builder().domain(State).representation(Bits2).codes(0, 1).create()


def snapshot(literal):
    """A literal's snapshot, as plain data."""
    return Plain.ToPlain(E.OfLiteral.Schema, literal)["objects"]["s0"]


def ev(name, *arguments, scope=None):
    """The value of a Basic operation."""
    return V.OfAny(E.operation(name, *arguments), scope)


L = E.literal
Nib = D.OfBits.Data(4)
U = D.OfInteger.Data(None, False)

## DOM-01 · Value domains name standards, with only their parameters, are compared by structure and validated

In [ ]:
names = [(D.Bool, "bool"), (D.Int, "int"), (I8, "int8"), (U8, "uint8"), (D.OfInteger.Builder().signed(False).create(), "uint"),
         (D.Float, "float"), (F32, "binary32"), (D.OfIeee754.Builder().rounding("roundTowardZero").create(),
                                                 "binary64 roundTowardZero"),
         (Bits2, "bits2"), (D.Bytes, "bytes"), (D.OfBytes.Builder().width(4).create(), "bytes4"), (D.Str, "str"),
         (D.OfIeee1164.Builder().create(), "std_logic"), (State, "enum(IDLE, RUN)"), (Coded, "packed(enum(IDLE, RUN) as bits2)")]
assert [(domain.name(), repr(domain)) for domain, _ in names] == [(name, name) for _, name in names]
assert all(domain.validate() == [] for domain, _ in names)
# The natives' defaults, compared by structure: an equal domain built anew is the same domain.
assert D.OfInteger.Data() == D.Int and D.OfIeee754.Data("binary64") == D.Float and D.OfUnicode.Data() == D.Str
assert D.OfBytes.Data() == D.Bytes and D.OfBool.Data() == D.Bool and I8 != U8 and I8 != D.Int
assert D.OfInteger.Builder().width(8).create() == I8 and repr(D.OfInteger) == "<domain kind integer>"
assert all(kind.Schema.validate() == [] for kind in [D.OfBool, D.OfInteger, D.OfIeee754, D.OfBits, D.OfBytes, D.OfUnicode,
                                                     D.OfIeee1164, D.OfEnum, D.OfPacked]) and D.Schema.validate() == []
assert [b.name for b in D.Schema.branches] == ["bool", "integer", "ieee754", "bits", "bytes", "unicode", "ieee1164", "enum",
                                               "packed", "named"]
assert D.FORMATS[2] == "binary64" and D.ROUNDINGS[0] == "roundTiesToEven" and D.OVERFLOWS == ("wrap", "saturate", "raise")
assert len(D.STATES) == 9
# Each kind's problems.
problems = [
    (D.OfInteger.Data(0), ["a width must be a positive int, got 0"]),
    (D.OfInteger.Data(None, False, "wrap"), ["an integer without a width cannot wrap"]),
    (D.OfInteger.Data(8, "yes", "clip"), ["signed must be a bool, got 'yes'", "overflow must be one of wrap, saturate, raise, got 'clip'"]),
    (D.OfIeee754.Data("binary8", "up"), [
        "format must be one of binary16, binary32, binary64, binary128, decimal64, decimal128, got 'binary8'",
        "rounding must be one of roundTiesToEven, roundTiesToAway, roundTowardPositive, roundTowardNegative, "
        "roundTowardZero, got 'up'"]),
    (D.OfBits.Data(), ["a bits domain needs a width"]),
    (D.OfBits.Data(0), ["a width must be a positive int, got 0"]),
    (D.OfBytes.Data(-1), ["a width must be a positive int, got -1"]),
    (D.OfEnum.Data(), ["an enum needs a member"]),
    (D.OfEnum.Data(("A", "", "A")), ["member 1 has no name", "member 'A' appears twice"]),
    (D.OfPacked.Data(I8, Bits2, ()), ["a packed domain packs an enum, got int8"]),
    (D.OfPacked.Data(), ["a packed domain packs an enum, got None"]),
    (D.OfPacked.Data(State, D.Bytes, (0, 1)), ["a packed domain's representation is an integer or bits of a width, got bytes"]),
    (D.OfPacked.Data(State, D.Int, (0, 1)), ["a packed domain's representation is an integer or bits of a width, got int"]),
    (D.OfPacked.Data(State, Bits2, (0,)), ["a packed domain needs a code per member: 2 members, 1 codes"]),
    (D.OfPacked.Data(State, D.OfBits.Data(1), (0, 2)), ["code 1 does not fit bits1: 2"]),
    (D.OfPacked.Data(State, Bits2, (1, 1)), ["code 1 is used twice"]),
    (D.OfPacked.Data(State, D.OfInteger.Data(2), (-2, 2)), ["code 1 does not fit int2: 2"]),
    (D.OfPacked.Data(State, D.OfInteger.Data(2), ("0", 1)), ["code 0 does not fit int2: '0'"]),
    (D.OfPacked.Data(D.OfEnum.Data(), D.OfInteger.Data(2, True, "x"), ()), [
        "domain: an enum needs a member", "representation: overflow must be one of wrap, saturate, raise, got 'x'"]),
]
for domain, expected in problems:
    assert domain.validate() == expected, (domain, domain.validate())
assert D.OfPacked.Data().name() == "packed(None as None)"
# Builders finalize as every builder does, and set each parameter.
built = D.OfIeee754.Builder(F32).rounding("roundTowardNegative").clone()
assert built == D.OfIeee754.Data("binary32", "roundTowardNegative") and F32.rounding == "roundTiesToEven"
assert D.OfBytes.Builder(D.OfBytes.Data()).width(3).update().width == 3
for builder in [D.OfBool.Builder(), D.OfUnicode.Builder(), D.OfIeee1164.Builder()]:
    assert builder.create().validate() == []
with raises(ValueError, match="create() is only valid without a source instance; use clone() or update()"):
    D.OfInteger.Builder(I8).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    D.OfInteger.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    D.OfInteger.Builder().update()
with raises(TypeError, match="expected integer data to build from, got bits2"):
    D.OfInteger.Builder(Bits2)

## DOM-02 · A domain holds the values of its native type that fit it, and includes only itself

In [ ]:
holds = [
    (I8, [127, -128], [128, -129, True, 1.0]), (U8, [0, 255], [-1, 256]), (D.Int, [10 ** 30, -5], [False]),
    (D.OfInteger.Data(signed=False), [10 ** 30], [-1]), (D.OfInteger.Data(0), [10 ** 30], []),  # not a width: unbounded
    (F32, [1.5, float("nan")], [1, "1.5"]), (D.OfIeee754.Data("decimal64"), ["0.1"], [0.1]),
    (D.OfBits.Data(3), [b"\x07", b"\x00"], [b"\x08", b"\x00\x07", "7"]), (D.OfBits.Data(8), [b"\xff"], [b""]),
    (D.OfBits.Data(9), [b"\x01\xff"], [b"\x02\x00", b"\xff"]), (D.OfBits.Data(), [], [b"\x00"]),
    (D.OfBytes.Data(2), [b"ab"], [b"a"]), (D.Bytes, [b"", b"abc"], ["abc"]),
    (D.Str, ["", "日本語 🚀"], ["\ud800", b"a"]), (D.OfIeee1164.Data(), ["Z", "-", "U"], ["z", "2", 1]),
    (State, ["RUN"], ["STOP", 0]), (Coded, ["IDLE"], ["STOP"]), (D.OfPacked.Data(), [], ["IDLE"]),
    (D.Bool, [True, False], [1, 0]),
]
for domain, inside, outside in holds:
    assert all(domain.contains(v) for v in inside) and not any(domain.contains(v) for v in outside), domain
assert D.OfIeee754.Data("decimal128").native() is str and F32.native() is float and I8.native() is int
# A domain includes only domains equal to it: Basic has no implied promotions.
assert I8.includes(D.OfInteger.Data(8)) and not I8.includes(D.Int) and not D.Int.includes(I8) and not I8.includes(U8)
assert I8.includes(FD.OfUnion(I8, D.OfInteger.Data(8))) and not I8.includes(FD.OfUnion(I8, U8))
assert FD.overlaps(FD.Anything, I8) and not D.Float.includes(F32)
# A native's default domain, and the domains of other values.
assert [D.of(v) for v in [True, 1, 1.5, "a", b"a"]] == [D.Bool, D.Int, D.Float, D.Str, D.Bytes]
assert D.of(E.variable("x").data) is D.Object
with raises(TypeError, match="a NoneType is not a value of the Basic dialect"):
    D.of(None)

## DOM-03 · Domains are registered by name, once, and written by name; any other by value

In [ ]:
D.register("uint8", U8)
assert D.registered("uint8") is U8 and D.name_of(U8) == "uint8" and D.name_of(D.OfInteger.Data(8, False, "wrap")) == "uint8"
assert D.name_of(I8) is None
with raises(ValueError, match="domain 'uint8' is already registered"):
    D.register("uint8", I8)
with raises(ValueError, match="uint8 is already registered as 'uint8'"):
    D.register("byte", D.OfInteger.Data(8, False, "wrap"))
D.register("uint16", D.OfInteger.Data(16, False))  # another domain: each name, and each domain, once
assert D.name_of(D.OfInteger.Data(16, False)) == "uint16"
with raises(LookupError, match="no domain registered as 'nope'"):
    D.registered("nope")
assert D.to_plain(U8) == {"named": {"name": "uint8"}}
assert D.to_plain(I8) == {"integer": {"width": 8, "signed": True, "overflow": "raise"}}
assert D.to_plain(D.Int) == {"integer": {"signed": True, "overflow": "raise"}}
assert D.to_plain(D.OfPacked.Data(State, U8, (0, 1))) == {
    "packed": {"domain": {"enum": {"members": ["IDLE", "RUN"]}}, "representation": {"named": {"name": "uint8"}}, "codes": [0, 1]}}
assert D.to_plain(D.OfUnicode.Data()) == {"unicode": {}}
for domain in [U8, I8, F32, Bits2, D.Bytes, D.Str, State, Coded, D.OfPacked.Data(State, U8, (0, 1)), D.Bool]:
    assert D.from_plain(D.to_plain(domain)) == domain
with raises(TypeError, match="not a domain: 'int8'"):
    D.to_plain("int8")
with raises(LookupError, match="no domain registered as 'nope'"):
    D.from_plain({"named": {"name": "nope"}})

## DOM-04 · A literal carries its domain; a native's default domain is no domain at all

In [ ]:
lit = E.literal(200, U8).data
assert lit.domain is U8 and lit.typed() is U8 and lit.validate() == []
assert E.literal(5, D.Int).data.domain is None and E.literal(5).data.typed() is None  # the default is left out
assert E.OfLiteral.Builder().value(-1).domain(I8).create().domain == I8
assert E.OfLiteral.Builder(lit).value(7).clone().domain is U8 and E.OfLiteral.Builder(lit).domain(None).clone().domain is None
assert E.OfLiteral.Builder(E.literal(1).data).domain(D.Int).update().domain is None
# A literal's domain is valid and holds its value.
assert E.literal(300, U8).data.validate() == ["a literal of uint8 cannot hold 300"]
assert E.literal(b"\x00", D.OfBits.Data()).data.validate() == ["domain: a bits domain needs a width"]
assert E.OfLiteral.Data(1, "int8").validate() == ["a literal's domain must be a value domain, got str"]
assert E.operation("eq", E.literal("RUN", Coded), E.literal("STOP", Coded)).data.validate() == [
    "argument 1: a literal of packed(enum(IDLE, RUN) as bits2) cannot hold 'STOP'"]
# Inference gives a literal's domain, and Basic does not promote: int8 and int8 add, int8 and int16 do not.
assert E.DIALECT.infer(lit) is U8 and E.DIALECT.infer(E.literal(1)) is D.Int
assert E.DIALECT.infer(E.operation("add", E.literal(1, I8), E.literal(2, I8))) == I8
with raises(TypeError, match="add cannot take (int8, int16); it takes (T, T) -> T for T in int | float or any integer or "
                             "ieee754 domain"):
    E.DIALECT.infer(E.operation("add", E.literal(1, I8), E.literal(2, D.OfInteger.Data(16))))
# Forms, `make` and `same` carry the domain.
assert lit.form().attributes == {"value": 200, "domain": U8}
assert F.same(E.DIALECT.make(lit.form()), lit) and F.same(lit, E.literal(200, D.OfInteger.Data(8, False, "wrap")).data)
assert not F.same(lit, E.literal(200, I8).data) and not F.same(lit, E.literal(200).data)
# A literal of another domain evaluates to a typed value; an evaluator without typed values refuses it.
assert V.OfAny(lit) == D.Value(U8, 200)
with raises(NotImplementedError, match="this evaluator has no values of uint8"):
    FE.Interpreter(E.DIALECT, {"operation": V.OPERATIONS})(lit)
assert V.OfAny(E.operation("add", E.literal(1, D.Int), 2)) == 3
# Translation keeps a domain only where the other dialect has one: none does yet.
with raises(ValueError, match="Basic literal 200 of uint8 has no Python counterpart"):
    T.between(E.DIALECT, P.DIALECT).forward(lit)

## DOM-05 · Over the wire a literal's domain is written by name when registered, otherwise by value

In [ ]:
assert snapshot(E.literal(200, U8).data) == {"kind": "literal", "int": 200, "domain": {"named": {"name": "uint8"}}}
assert snapshot(E.literal(-1, I8).data) == {
    "kind": "literal", "int": -1, "domain": {"integer": {"width": 8, "signed": True, "overflow": "raise"}}}
assert snapshot(E.literal(5, D.Int).data) == {"kind": "literal", "int": 5}
text = JSON.ToJSON(E.OfLiteral.Schema, E.literal("RUN", Coded).data)
assert text == ('{"root": "s0", "objects": {"s0": {"kind": "literal", "str": "RUN", "domain": {"packed": {"domain": '
                '{"enum": {"members": ["IDLE", "RUN"]}}, "representation": {"bits": {"width": 2}}, "codes": [0, 1]}}}}}')
for literal in [E.literal(200, U8).data, E.literal(-1, I8).data, E.literal("RUN", Coded).data, E.literal(1.5, F32).data,
                E.literal("Z", D.OfIeee1164.Data()).data, E.literal(b"\x03", Bits2).data]:
    back = JSON.FromJSON(E.Builders)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, literal))
    assert F.same(back, literal) and back.domain == literal.domain
    assert F.same(YAML.FromYAML(E.Builders)(E.OfLiteral.Schema, YAML.ToYAML(E.OfLiteral.Schema, literal)), literal)
    assert Validators.Validate(E.Builders)(E.OfLiteral.Schema, back) == []
assert JSON.FromJSON(E.Builders)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, E.literal(1, U8).data)).domain is U8
with raises(LookupError, match="no domain registered as 'nope'"):
    Plain.FromPlain(E.Builders)(E.OfLiteral.Schema, {"root": "s0", "objects": {"s0": {
        "kind": "literal", "int": 1, "domain": {"named": {"name": "nope"}}}}})
# Literals compare under their meta-schema with their domains.
compare = lambda a, b: C.OfObject(E.OfLiteral.Schema, a).compare(C.OfObject(E.OfLiteral.Schema, b))  # noqa: E731
assert compare(E.literal(1, I8).data, E.literal(1, D.OfInteger.Data(8)).data) == 0
assert compare(E.literal(1, I8).data, E.literal(1, U8).data) is None and compare(E.literal(1, I8).data, E.literal(1).data) is None
# The builder reads and writes the domain through the visitor protocols too.
def domain_field(builder, branch, name, act):
    """Calls `act` with the native handle of the domain's field `name`, in the union branch `branch`."""
    builder.property("domain", lambda p: p.value(lambda a: a.as_union(lambda u: u.property(branch, lambda q: q.value(
        lambda b: b.as_object(lambda o: o.property(name, lambda w: w.value(lambda n: n.as_native(act)))))))))


builder = E.OfLiteral.Builder(E.literal(-1, I8).data)
names = []
builder.properties(lambda p: names.append(p.name()))
assert builder.has("domain") and names == ["kind", "int", "domain"]
widths = []
domain_field(builder, "integer", "width", lambda n: widths.append(n.get()))
assert widths == [8]
domain_field(builder, "integer", "width", lambda n: n.set(16))
assert builder.clone().domain == D.OfInteger.Data(16)
builder.property("domain", lambda p: p.clear())
assert not builder.has("domain") and builder.clone().domain is None
domain_field(builder, "named", "name", lambda n: n.set("uint8"))
assert builder.clone().domain is U8
builder.clear("domain")
assert builder.clone().domain is None
builder.property("domain", lambda p: p.value(lambda a: a.as_union(lambda u: None)))  # a union without a branch is no value
assert builder.clone().domain is None

## DOM-06 · A value of another domain is a typed value, compared only within its domain

In [ ]:
assert D.value(U8, 200) == D.Value(U8, 200) and D.value(D.Int, 5) == 5 and type(D.value(D.Bool, True)) is bool
assert D.of(D.Value(U8, 1)) is U8 and repr(D.Value(U8, 200)) == "200 as uint8"
with raises(ValueError, match="uint8 cannot hold 256"):
    D.Value(U8, 256)
with raises(TypeError, match="not a value domain: 'uint8'"):
    D.Value("uint8", 1)
# eq and ne within one domain; other domains, typed or default, are incomparable.
assert ev("eq", L(5, I8), L(5, I8)) is True and ev("ne", L(5, I8), L(6, I8)) is True
assert ev("eq", L(5, I8), 5) is None and ev("eq", L(5, I8), L(5, U8)) is None and ev("lt", L(5, I8), 6) is None
assert ev("lt", L(5, I8), L(5, U8)) is None
# Ordered: integers, IEEE 754 numbers, bytes and packed enums (by code); unordered: bits, enums and std_logic.
assert ev("lt", L(-1, I8), L(1, I8)) is True and ev("ge", L(1.5, F32), L(2.5, F32)) is False
Bytes2 = D.OfBytes.Data(2)
assert ev("lt", L(b"\x00\x01", Bytes2), L(b"\x01\x00", Bytes2)) is True
Reversed = D.OfPacked.Data(State, Bits2, (1, 0))
assert ev("gt", L("IDLE", Reversed), L("RUN", Reversed)) is True and ev("lt", L("IDLE", Coded), L("RUN", Coded)) is True
assert ev("eq", L("RUN", Coded), L("RUN", Coded)) is True
assert ev("lt", L(b"\x01", Bits2), L(b"\x02", Bits2)) is None and ev("eq", L(b"\x01", Bits2), L(b"\x01", Bits2)) is True
assert ev("lt", L("IDLE", State), L("RUN", State)) is None and ev("ne", L("IDLE", State), L("RUN", State)) is True
Logic = D.OfIeee1164.Data()
assert ev("eq", L("X", Logic), L("X", Logic)) is True and ev("le", L("0", Logic), L("1", Logic)) is None
# A format's floats compare as mbse-schemas compares floats: NaNs equal each other, and -0 comes before 0.
nan = float("nan")
assert ev("eq", L(nan, F32), L(nan, F32)) is True and ev("lt", L(nan, F32), L(1.0, F32)) is None
assert ev("lt", L(-0.0, F32), L(0.0, F32)) is True
assert ev("eq", L("1.5", D.OfIeee754.Data("decimal64")), L("1.5", D.OfIeee754.Data("decimal64"))) is True  # see DOM-12
# A scope may bind a variable to a typed value; unknown stays unknown.
assert V.OfAny(E.variable("x").eq(L(3, I8)), {"x": D.Value(I8, 3)}) is True
assert V.OfAny(E.variable("x").lt(L(3, I8)), {"x": None}) is None

## DOM-07 · Integer arithmetic is exact, then the domain's overflow wraps, saturates or raises

In [ ]:
assert ev("add", L(100, I8), L(27, I8)) == D.Value(I8, 127)
with raises(OverflowError, match="add overflows int8: 128"):
    ev("add", L(100, I8), L(28, I8))
with raises(OverflowError, match="neg overflows int8: 128"):
    ev("neg", L(-128, I8))
assert ev("add", L(250, U8), L(10, U8)) == D.Value(U8, 4) and ev("sub", L(0, U8), L(1, U8)) == D.Value(U8, 255)
W8 = D.OfInteger.Data(8, True, "wrap")
assert ev("mul", L(64, W8), L(2, W8)) == D.Value(W8, -128) and ev("neg", L(-128, W8)) == D.Value(W8, -128)
S8 = D.OfInteger.Data(8, True, "saturate")
assert ev("add", L(100, S8), L(100, S8)) == D.Value(S8, 127) and ev("sub", L(-100, S8), L(100, S8)) == D.Value(S8, -128)
# An unbounded unsigned integer has a least value only, and without a width wrap does not apply.
assert ev("add", L(10 ** 30, U), L(1, U)) == D.Value(U, 10 ** 30 + 1)
US = D.OfInteger.Data(None, False, "saturate")
assert ev("sub", L(1, US), L(2, US)) == D.Value(US, 0)
with raises(OverflowError, match="sub overflows uint: -1"):
    ev("sub", L(1, U), L(2, U))
UW = D.OfInteger.Data(None, False, "wrap")  # invalid, as validation reports; evaluation raises
with raises(OverflowError, match="sub overflows uint: -1"):
    ev("sub", L(1, UW), L(2, UW))
# Operands of different domains raise: Basic does not promote.
with raises(TypeError, match="add expects numbers of one domain, got int8 and int"):
    ev("add", L(1, I8), 1)
with raises(TypeError, match="add expects numbers of one domain, got int8 and uint8"):
    ev("add", L(1, I8), L(1, U8))
with raises(TypeError, match="neg expects a number, got enum(IDLE, RUN)"):
    ev("neg", L("IDLE", State))
# The default domains' arithmetic is unchanged; IEEE 754 formats round in their own (see DOM-11).
assert ev("add", 2 ** 70, 1) == 2 ** 70 + 1 and ev("mul", 1.5, 2.0) == 3.0
assert ev("add", L(1.5, F32), L(1.5, F32)) == D.Value(F32, 3.0)

## DOM-08 · The bitwise operations take integers or bits of one domain, and shifts a count of any integer domain

In [ ]:
assert ev("bitand", 12, 10) == 8 and ev("bitor", 12, 10) == 14 and ev("bitxor", 12, 10) == 6 and ev("bitnot", 5) == -6
assert ev("bitnot", L(5, U8)) == D.Value(U8, 250) and ev("bitnot", L(5, I8)) == D.Value(I8, -6)
assert ev("bitand", L(-1, I8), L(15, I8)) == D.Value(I8, 15)
assert ev("bitxor", L(b"\x0c", Nib), L(b"\x0a", Nib)) == D.Value(Nib, b"\x06")
assert ev("bitor", L(b"\x0c", Nib), L(b"\x03", Nib)) == D.Value(Nib, b"\x0f")
assert ev("bitand", L(b"\x0c", Nib), L(b"\x0a", Nib)) == D.Value(Nib, b"\x08")
assert ev("bitnot", L(b"\x05", Nib)) == D.Value(Nib, b"\x0a")  # within the width
# Without a width, a result outside the domain applies its overflow.
with raises(OverflowError, match="bitnot overflows uint: -6"):
    ev("bitnot", L(5, U))
with raises(TypeError, match="bitand expects integers or bits of one domain, got int8 and bits4"):
    ev("bitand", L(1, I8), L(b"\x01", Nib))
with raises(TypeError, match="bitnot expects an integer or bits, got bool"):
    ev("bitnot", True)
# Shifts: on integers by a power of two, with the overflow (shr toward negative infinity); on bits, logical.
assert ev("shl", 3, 4) == 48 and ev("shr", -7, 1) == -4 and ev("shr", L(-128, I8), L(7, U8)) == D.Value(I8, -1)
assert ev("shl", L(1, U8), 7) == D.Value(U8, 128) and ev("shl", L(1, U8), 8) == D.Value(U8, 0)
with raises(OverflowError, match="shl overflows int8: 128"):
    ev("shl", L(1, I8), 7)
assert ev("shl", L(b"\x09", Nib), 1) == D.Value(Nib, b"\x02") and ev("shr", L(b"\x09", Nib), 3) == D.Value(Nib, b"\x01")
with raises(ValueError, match="shl needs a non-negative count, got -1"):
    ev("shl", 1, -1)
with raises(TypeError, match="shr expects an integer count, got float"):
    ev("shr", 1, 1.0)
with raises(TypeError, match="shl expects an integer or bits, got str"):
    ev("shl", "a", 1)
x = E.variable("x")
assert ev("bitand", x, 1, scope={"x": None}) is None and ev("shl", x, 1, scope={"x": None}) is None
assert ev("shl", 1, x, scope={"x": None}) is None

## DOM-09 · Signatures take any domain of a family, so inference follows typed values

In [ ]:
infer = E.DIALECT.infer
assert infer(E.operation("eq", L("IDLE", State), L("RUN", State))) is D.Bool
assert infer(E.operation("lt", L(1, I8), L(2, I8))) is D.Bool
with raises(TypeError, match="lt cannot take (enum(IDLE, RUN), enum(IDLE, RUN)); it takes (T, T) -> bool for T in "
                             "int | float | str | bytes or any integer, ieee754, bytes, unicode or packed domain"):
    infer(E.operation("lt", L("IDLE", State), L("RUN", State)))
assert infer(E.operation("mul", L(1.5, F32), x), {"x": FD.Anything}) == F32  # an unknown takes the other's domain
assert infer(E.operation("bitxor", L(b"\x01", Nib), L(b"\x02", Nib))) == Nib and infer(E.operation("bitnot", 1)) is D.Int
assert infer(E.operation("shl", L(1, U8), L(2, I8))) == U8 and infer(E.operation("shr", x, 1), {"x": FD.Anything}) is D.Int
assert infer(E.operation("shl", 1, x), {"x": FD.OfUnion(D.Int, I8)}) is D.Int
with raises(TypeError, match="shl cannot take (int, float); it takes (T, any integer domain) -> T for T in int or any "
                             "integer or bits domain"):
    infer(E.operation("shl", 1, 1.5))
with raises(TypeError, match="shr cannot take (float, int)"):
    infer(E.operation("shr", 1.5, 1))
assert E.operation("bitand", 1, 2).data.validate(core=True) == []
# A family includes every domain of its kinds, and holds their values.
family = D.SIGNATURES["add"].family
assert repr(family) == "integer or ieee754 domain" and family.contains(D.Value(I8, 1)) and family.contains(1.5)
assert not family.contains("a") and not family.contains(D.Value(State, "RUN"))

## DOM-10 · An IEEE 754 value is a float up to binary64 and canonical text otherwise, which the format must hold

In [ ]:
holds = [
    (F16, [1.5, 65504.0, 2.0 ** -24, -0.0, math.inf, math.nan], [0.1, 1e10, 2.0 ** -25, "1.5"]),
    (F32, [0.10000000149011612], [0.1]),
    (D.OfIeee754.Data("binary64", "roundTowardZero"), [0.1, 5e-324], ["0.1"]),
    (F128, ["0.1", "1E+2", "-0", "Infinity", "-Infinity", "NaN", "1.189731495357231765085759326628007E+4932"],
     ["0.10", "100", "1e2", "-NaN", "1E+5000", "0.1 ", 0.1]),
    (D64, ["1.50", "1.000000000000000E+16", "0E-398", "-0", "0.000001", "1E-7", "NaN"],
     ["1.5E+0", "12345678901234567", "1E+385", "1E-399", "0.0000001", "+1", "abc"]),
    (D128, ["1.000000000000000000000000000000000", "2E+6111"], ["1E+6112"]),
]
for domain, inside, outside in holds:
    assert all(domain.contains(v) for v in inside) and not any(domain.contains(v) for v in outside), domain
assert F128.native() is str and D64.native() is str and F16.native() is float
assert not D.OfIeee754.Data("binary8").contains(1.5)  # an unknown format holds nothing
# A decimal value's text is General Decimal Arithmetic's scientific form, keeping the coefficient and the exponent.
data = [(False, 15, -1), (True, 150, -2), (False, 1, 2), (False, 123, -9), (False, 0, 3), (True, 0, 0)]
assert [Ieee754.text(Ieee754.Datum(None, *d)) for d in data] == ["1.5", "-1.50", "1E+2", "1.23E-7", "0E+3", "-0"]
assert Ieee754.text(Ieee754.NAN) == "NaN" and Ieee754.FORMATS["decimal64"] == (10, 16, 384)

## DOM-11 · IEEE 754 arithmetic is exact, then rounded to the format by the domain's rounding direction

In [ ]:
EVEN, AWAY, UP, DOWN, ZERO = D.ROUNDINGS
inf, nan = math.inf, math.nan
TABLE = [  # format, rounding, operation, operands, result
    ("binary16", EVEN, "add", 1.0, 0.00048828125, 1.0),
    ("binary16", EVEN, "add", 1.0009765625, 0.00048828125, 1.001953125),
    ("binary16", AWAY, "add", 1.0, 0.00048828125, 1.0009765625),
    ("binary16", UP, "add", 1.0, 5.960464477539063e-08, 1.0009765625),
    ("binary16", DOWN, "add", 1.0, 5.960464477539063e-08, 1.0),
    ("binary16", ZERO, "add", 1.0, 5.960464477539063e-08, 1.0),
    ("binary16", DOWN, "add", -1.0, -5.960464477539063e-08, -1.0009765625),
    ("binary16", UP, "add", -1.0, -5.960464477539063e-08, -1.0),
    ("binary16", EVEN, "add", 65504.0, 65504.0, inf),
    ("binary16", ZERO, "add", 65504.0, 65504.0, 65504.0),
    ("binary16", UP, "add", 65504.0, 65504.0, inf),
    ("binary16", DOWN, "add", 65504.0, 65504.0, 65504.0),
    ("binary16", DOWN, "add", -65504.0, -65504.0, -inf),
    ("binary16", UP, "add", -65504.0, -65504.0, -65504.0),
    ("binary16", AWAY, "mul", 65504.0, 2.0, inf),
    ("binary16", EVEN, "mul", 6.103515625e-05, 0.0009765625, 5.960464477539063e-08),
    ("binary16", EVEN, "mul", 5.960464477539063e-08, 0.5, 0.0),
    ("binary16", AWAY, "mul", 5.960464477539063e-08, 0.5, 5.960464477539063e-08),
    ("binary16", EVEN, "add", 2047.0, 0.5, 2048.0),
    ("binary16", EVEN, "sub", 1.0, 1.0, 0.0),
    ("binary16", DOWN, "sub", 1.0, 1.0, -0.0),
    ("binary16", EVEN, "add", -0.0, -0.0, -0.0),
    ("binary16", EVEN, "add", 0.0, -0.0, 0.0),
    ("binary16", EVEN, "mul", -0.0, 5.0, -0.0),
    ("binary16", EVEN, "sub", inf, inf, nan),
    ("binary16", EVEN, "add", inf, 1.0, inf),
    ("binary16", EVEN, "add", 1.0, -inf, -inf),
    ("binary16", EVEN, "add", nan, 1.0, nan),
    ("binary16", EVEN, "mul", inf, 0.0, nan),
    ("binary16", EVEN, "mul", 0.0, -inf, nan),
    ("binary16", EVEN, "mul", inf, -2.0, -inf),
    ("binary16", EVEN, "mul", nan, 1.0, nan),
    ("binary16", EVEN, "neg", 1.5, None, -1.5),
    ("binary16", EVEN, "neg", nan, None, nan),
    ("binary16", EVEN, "neg", -0.0, None, 0.0),
    ("binary32", EVEN, "add", 16777216.0, 1.0, 16777216.0),
    ("binary32", UP, "add", 16777216.0, 1.0, 16777218.0),
    ("binary32", EVEN, "mul", 0.10000000149011612, 3.0, 0.30000001192092896),
    ("binary64", ZERO, "add", 0.1, 0.2, 0.3),
    ("binary64", UP, "add", 0.1, 0.2, 0.30000000000000004),
    ("binary64", DOWN, "mul", 0.3333333333333333, 3.0, 0.9999999999999999),
    ("binary128", EVEN, "add", "0.1", "0.2", "0.30000000000000000000000000000000004"),
    ("binary128", EVEN, "mul", "1E+4932", "1E+1", "Infinity"),
    ("binary128", ZERO, "mul", "1E+4932", "1E+1", "1.189731495357231765085759326628007E+4932"),
    ("binary128", EVEN, "sub", "1", "1", "0"),
    ("binary128", DOWN, "sub", "1", "1", "-0"),
    ("binary128", EVEN, "neg", "-0", None, "0"),
    ("binary128", EVEN, "add", "Infinity", "1", "Infinity"),
    ("binary128", EVEN, "mul", "1E-4960", "1E-10", "0"),
    ("decimal64", EVEN, "add", "1.5", "1.50", "3.00"),
    ("decimal64", EVEN, "mul", "1.5", "1.50", "2.250"),
    ("decimal64", EVEN, "add", "9999999999999999", "1", "1.000000000000000E+16"),
    ("decimal64", EVEN, "add", "9999999999999999", "0.5", "1.000000000000000E+16"),
    ("decimal64", AWAY, "add", "1000000000000000", "0.5", "1000000000000001"),
    ("decimal64", EVEN, "add", "1000000000000000", "0.5", "1000000000000000"),
    ("decimal64", UP, "add", "1000000000000000", "0.1", "1000000000000001"),
    ("decimal64", DOWN, "add", "-1000000000000000", "-0.1", "-1000000000000001"),
    ("decimal64", ZERO, "add", "-1000000000000000", "-0.9", "-1000000000000000"),
    ("decimal64", EVEN, "mul", "1E+369", "1E+10", "1.0000000000E+379"),
    ("decimal64", EVEN, "add", "9.999999999999999E+384", "1E+369", "Infinity"),
    ("decimal64", ZERO, "add", "9.999999999999999E+384", "1E+369", "9.999999999999999E+384"),
    ("decimal64", EVEN, "mul", "1E-398", "0.1", "0E-398"),
    ("decimal64", UP, "mul", "1E-398", "0.1", "1E-398"),
    ("decimal64", EVEN, "mul", "1E-390", "0.00001", "1E-395"),
    ("decimal64", EVEN, "sub", "1.5", "1.5", "0.0"),
    ("decimal64", DOWN, "sub", "1.5", "1.5", "-0.0"),
    ("decimal64", EVEN, "add", "-0", "-0", "-0"),
    ("decimal64", EVEN, "mul", "-0.0", "1E+5", "-0E+4"),
    ("decimal64", EVEN, "mul", "1E+369", "0E+10", "0E+369"),
    ("decimal64", EVEN, "add", "NaN", "1", "NaN"),
    ("decimal64", EVEN, "mul", "Infinity", "0", "NaN"),
    ("decimal64", EVEN, "neg", "1.50", None, "-1.50"),
    ("decimal64", EVEN, "add", "0.000001", "1E-7", "0.0000011"),
    ("decimal64", EVEN, "mul", "123", "1E+5", "1.23E+7"),
    ("decimal128", EVEN, "mul", "0.1", "3", "0.3"),
    ("decimal128", EVEN, "add", "1E+6111", "1E+6111", "2E+6111"),
    ("decimal128", EVEN, "add", "1", "1E-40", "1.000000000000000000000000000000000"),
]
CONTEXTS = {"decimal64": (16, 384), "decimal128": (34, 6144)}
MODES = dict(zip(D.ROUNDINGS, [decimal.ROUND_HALF_EVEN, decimal.ROUND_HALF_UP, decimal.ROUND_CEILING, decimal.ROUND_FLOOR,
                               decimal.ROUND_DOWN]))


def alike(x, y):
    """The same value: floats by sign too, NaNs alike."""
    if isinstance(x, float) and isinstance(y, float):
        return (math.isnan(x) and math.isnan(y)) or (x == y and math.copysign(1, x) == math.copysign(1, y))
    return x == y


for format, rounding, operation, a, b, expected in TABLE:
    domain = D.OfIeee754.Data(format, rounding)
    result = ev(operation, *[L(v, domain) for v in (a, b) if v is not None])
    assert isinstance(result, D.Value) and result.domain == domain and alike(result.value, expected), (format, a, b, result)
    # Python's decimal module and the host's floats agree.
    if format in CONTEXTS:
        precision, emax = CONTEXTS[format]
        context = decimal.Context(prec=precision, Emax=emax, Emin=1 - emax, clamp=1, rounding=MODES[rounding], traps=[])
        left, right = decimal.Decimal(a), decimal.Decimal(b or 0)
        negate = lambda value, _: value.copy_negate()  # noqa: E731
        reference = {"add": context.add, "sub": context.subtract, "mul": context.multiply}.get(operation, negate)(left, right)
        assert ("NaN" if reference.is_nan() else str(reference)) == expected, (format, a, b, reference)
    elif rounding == EVEN and format in ("binary16", "binary32") and operation != "neg":
        exact = {"add": a + b, "sub": a - b, "mul": a * b}[operation]
        code = "e" if format == "binary16" else "f"
        try:
            reference = struct.unpack(code, struct.pack(code, exact))[0]
        except OverflowError:
            reference = math.copysign(math.inf, exact)
        assert alike(reference, expected), (format, a, b, reference)
# The default float keeps the host's arithmetic, which binary64 rounding ties to even is.
assert ev("add", 0.1, 0.2) == 0.30000000000000004 and D.OfIeee754.Data("binary64") == D.Float

## DOM-12 · IEEE 754 values compare by totalOrder, cohorts by exponent, and NaNs equal each other

In [ ]:
assert ev("eq", L("1.5", D64), L("1.5", D64)) is True and ev("eq", L("1.5", D64), L("1.50", D64)) is False
assert ev("lt", L("1.50", D64), L("1.5", D64)) is True and ev("lt", L("-1.5", D64), L("-1.50", D64)) is True
assert ev("lt", L("1.5", D64), L("2", D64)) is True and ev("gt", L("-1", D64), L("-2", D64)) is True
assert ev("lt", L("-0", D64), L("0", D64)) is True and ev("lt", L("-Infinity", D64), L("-9.999999999999999E+384", D64)) is True
assert ev("lt", L("9.999999999999999E+384", D64), L("Infinity", D64)) is True and ev("eq", L("Infinity", D64), L("Infinity", D64)) is True
assert ev("eq", L("NaN", D64), L("NaN", D64)) is True and ev("lt", L("NaN", D64), L("1", D64)) is None
assert ev("eq", L("1", D64), L("NaN", D64)) is False and ev("gt", L("1", D64), L("-1", D64)) is True
assert ev("lt", L("0.1", F128), L("0.2", F128)) is True and ev("eq", L("0.1", F128), L("0.1", F128)) is True
assert ev("le", L(-0.0, F16), L(0.0, F16)) is True and ev("eq", L(math.nan, F16), L(math.nan, F16)) is True
# A binary128 value's text is the shortest that rounds back to it.
assert ev("mul", L("3", F128), L("0.5", F128)) == D.Value(F128, "1.5")
assert ev("add", L("0.1", F128), L("0.2", F128)) == D.Value(F128, "0.30000000000000000000000000000000004")
assert ev("mul", L("0.5", F128), L("0.5", F128)) == D.Value(F128, "0.25") and ev("neg", L("0.49", F128)) == D.Value(F128, "-0.49")
assert ev("mul", L("1E+48", F128), L("1E+1", F128)) == D.Value(F128, "1E+49")  # the nearest is below 1E+49

## DOM-13 · An operation's domain is its result's: the conversions need one, and other core operations take none

In [ ]:
conversion = L(1).convert(I8).data
assert conversion.domain is I8 and conversion.typed() is I8 and conversion.validate() == []
assert E.operation("convert", 1).data.validate() == ["convert needs a domain"]
assert E.OfOperation.Builder().name("add").arguments(1, 2).domain(I8).create().validate() == ["add takes no domain"]
assert E.OfOperation.Builder().name("pack").arguments(L("RUN", Coded)).domain(Bits2).create().validate() == ["pack takes no domain"]
assert E.OfOperation.Builder().name("cast").arguments(1).domain(I8).create().validate() == []  # an extension may take one
assert E.OfOperation.Data("convert", (L(1).data,), "int8").validate() == ["an operation's domain must be a value domain, got str"]
assert E.OfOperation.Data("convert", (L(1).data,), D.OfBits.Data()).validate() == ["domain: a bits domain needs a width"]
# Kept when it is a native's default, and written as a literal's is: by name when registered, otherwise by value.
assert L(5, I8).convert(D.Int).data.domain == D.Int
assert Plain.ToPlain(E.OfOperation.Schema, L(1).convert(U8).data)["objects"]["s0"]["domain"] == {"named": {"name": "uint8"}}
operation = L(300).convert(U8).data
back = JSON.FromJSON(E.Builders).Reachable(E.OfOperation.Schema, JSON.ToJSON.Reachable(E.OfOperation.Schema, operation))
assert back.domain == U8 and F.same(back, operation)
# Inference gives the operation's own domain; pack gives a packed domain's representation.
assert E.DIALECT.infer(L(1).convert(I8)) == I8 and E.DIALECT.infer(L("RUN", Coded).pack()) == Bits2
assert E.DIALECT.infer(E.operation("pack", x), {"x": FD.Anything}) is FD.Anything
with raises(TypeError, match="pack cannot take (int); it takes (packed) -> its representation"):
    E.DIALECT.infer(E.operation("pack", 1))
with raises(TypeError, match="convert cannot take (object); it takes (T) -> the operation's domain"):
    E.DIALECT.infer(E.variable("o").convert(I8), {"o": D.Object})
# Forms and `make` carry it, and terms write the conversions and the bitwise operations.
assert conversion.form().attributes == {"name": "convert", "domain": I8} and F.same(E.DIALECT.make(conversion.form()), conversion)
terms = [x.bitand(1), x.bitor(1), x.bitxor(1), x.bitnot(), x.shl(1), x.shr(1), x.pack(), x.reinterpret(I8), x.unpack(Coded)]
assert [t.data.name for t in terms] == ["bitand", "bitor", "bitxor", "bitnot", "shl", "shr", "pack", "reinterpret", "unpack"]
assert terms[7].data.domain is I8 and terms[8].data.domain is Coded and terms[6].data.domain is None

## DOM-14 · convert keeps the value in the operation's domain, rounding and overflowing as that domain does

In [ ]:
S8, US = D.OfInteger.Data(8, True, "saturate"), D.OfInteger.Data(None, False, "saturate")
assert V.OfAny(L(300).convert(U8)) == D.Value(U8, 44) and V.OfAny(L(-1, I8).convert(D.Int)) == -1
assert V.OfAny(L(300).convert(S8)) == D.Value(S8, 127)
with raises(OverflowError, match="convert overflows int8: 300"):
    V.OfAny(L(300).convert(I8))
# An integer to IEEE 754, and IEEE 754 to IEEE 754, rounded by the target's direction.
assert V.OfAny(L(2 ** 24 + 1).convert(F32)) == D.Value(F32, 16777216.0)
UP32 = D.OfIeee754.Data("binary32", "roundTowardPositive")
assert V.OfAny(L(2 ** 24 + 1).convert(UP32)) == D.Value(UP32, 16777218.0) and V.OfAny(L(5).convert(D.Float)) == 5.0
assert V.OfAny(L(10 ** 20).convert(D64)) == D.Value(D64, "1.000000000000000E+20") and V.OfAny(L(-7).convert(D64)) == D.Value(D64, "-7")
assert V.OfAny(L(0.1).convert(F32)) == D.Value(F32, 0.10000000149011612) and V.OfAny(L(0.5).convert(D64)) == D.Value(D64, "0.5")
assert V.OfAny(L(100.0).convert(D64)) == D.Value(D64, "100") and V.OfAny(L(0.1).convert(D64)) == D.Value(D64, "0.1000000000000000")
assert V.OfAny(L("1.50", D64).convert(D128)) == D.Value(D128, "1.50") and V.OfAny(L("0.1", D64).convert(D.Float)) == 0.1
assert V.OfAny(L(math.nan).convert(F128)) == D.Value(F128, "NaN") and V.OfAny(L(-math.inf).convert(D64)) == D.Value(D64, "-Infinity")
assert V.OfAny(L(1e300).convert(F16)) == D.Value(F16, math.inf)
# IEEE 754 to an integer: rounded in the source's direction, then the target's overflow.
Z64 = D.OfIeee754.Data("binary64", "roundTowardZero")
assert V.OfAny(L(2.5).convert(D.Int)) == 2 and V.OfAny(L(-2.5).convert(D.Int)) == -2 and V.OfAny(L(2.7).convert(D.Int)) == 3
assert V.OfAny(L(2.7, Z64).convert(I8)) == D.Value(I8, 2) and V.OfAny(L(-2.7, Z64).convert(I8)) == D.Value(I8, -2)
assert V.OfAny(L("2.5", D64).convert(I8)) == D.Value(I8, 2) and V.OfAny(L(300.0).convert(U8)) == D.Value(U8, 44)
assert V.OfAny(L(math.inf).convert(S8)) == D.Value(S8, 127) and V.OfAny(L(-math.inf).convert(S8)) == D.Value(S8, -128)
assert V.OfAny(L(-math.inf).convert(US)) == D.Value(US, 0)
with raises(ValueError, match="NaN has no integer value"):
    V.OfAny(L(math.nan).convert(I8))
with raises(OverflowError, match="convert overflows int8: -Infinity"):
    V.OfAny(L(-math.inf).convert(I8))
with raises(OverflowError, match="convert overflows uint: Infinity"):
    V.OfAny(L(math.inf).convert(US))  # no greatest value to saturate to
# Bytes of one width to another when the value fits, and an enum to a packed domain of it and back.
B2, B4 = D.OfBytes.Data(2), D.OfBytes.Data(4)
assert V.OfAny(L(b"ab").convert(B2)) == D.Value(B2, b"ab") and V.OfAny(L(b"ab", B2).convert(D.Bytes)) == b"ab"
with raises(ValueError, match="bytes4 cannot hold b'ab'"):
    V.OfAny(L(b"ab").convert(B4))
assert V.OfAny(L("RUN", State).convert(Coded)) == D.Value(Coded, "RUN") and V.OfAny(L("RUN", Coded).convert(State)) == D.Value(State, "RUN")
# A domain to itself is no change; anything else raises.
assert V.OfAny(L(5, I8).convert(I8)) == D.Value(I8, 5)
Other = D.OfPacked.Data(D.OfEnum.Data(("A", "B")), Bits2, (0, 1))


class Point:
    """An object: it writes its properties through accept."""

    def accept(self, visitor):
        pass


for value, domain, message in [(L("5"), I8, "cannot convert str to int8"), (L(b"\x01", Bits2), I8, "cannot convert bits2 to int8"),
                               (L("RUN", State), Other, "cannot convert enum(IDLE, RUN) to packed(enum(A, B) as bits2)"),
                               (E.variable("o"), I8, "cannot convert Point to int8")]:
    with raises(TypeError, match=message):
        V.OfAny(value.convert(domain), {"o": Point()})
assert V.OfAny(x.convert(I8), {"x": None}) is None

## DOM-15 · reinterpret keeps the bit pattern, big-endian, between fixed-width domains of one size

In [ ]:
I64, U16, I12 = D.OfInteger.Data(64), D.OfInteger.Data(16, False), D.OfInteger.Data(12)
Bits64, Bits128, Bytes4 = D.OfBits.Data(64), D.OfBits.Data(128), D.OfBytes.Data(4)
assert V.OfAny(L(200, U8).reinterpret(I8)) == D.Value(I8, -56) and V.OfAny(L(-56, I8).reinterpret(U8)) == D.Value(U8, 200)
assert V.OfAny(L(1.5).reinterpret(I64)) == D.Value(I64, 4609434218613702656)
assert V.OfAny(L(4609434218613702656, I64).reinterpret(D.Float)) == 1.5
assert V.OfAny(L(-0.0).reinterpret(Bits64)) == D.Value(Bits64, b"\x80" + bytes(7))
assert V.OfAny(L(math.nan, F32).reinterpret(Bytes4)) == D.Value(Bytes4, b"\x7f\xc0\x00\x00")  # the canonical quiet NaN
assert V.OfAny(L(b"\x7f\x80\x00\x00", Bytes4).reinterpret(F32)) == D.Value(F32, math.inf)
assert math.isnan(V.OfAny(L(b"\xff\x80\x00\x01", Bytes4).reinterpret(F32)).value)
assert V.OfAny(L(1, U16).reinterpret(F16)) == D.Value(F16, 2.0 ** -24) and V.OfAny(L(65504.0, F16).reinterpret(U16)) == D.Value(U16, 0x7BFF)
assert V.OfAny(L(2.0 ** -24, F16).reinterpret(U16)) == D.Value(U16, 1) and V.OfAny(L(0.0, F16).reinterpret(U16)) == D.Value(U16, 0)
assert V.OfAny(L("1", F128).reinterpret(Bits128)) == D.Value(Bits128, bytes.fromhex("3fff" + "00" * 14))
assert V.OfAny(L(bytes.fromhex("3fff8" + "0" * 27), Bits128).reinterpret(F128)) == D.Value(F128, "1.5")
assert V.OfAny(L(-math.inf, F16).reinterpret(U16)) == D.Value(U16, 0xFC00)
assert V.OfAny(L(b"\x0f\xff", D.OfBits.Data(12)).reinterpret(I12)) == D.Value(I12, -1)
for value, domain, message in [(L(1, I8), D.OfBits.Data(16), "cannot reinterpret int8 as bits16: 8 bits and 16"),
                               (L("1", D64), I64, "cannot reinterpret decimal64 as int64"),
                               (L(1), I64, "cannot reinterpret int as int64"), (L(1, I8), D.Bytes, "cannot reinterpret int8 as bytes")]:
    with raises(TypeError, match=message):
        V.OfAny(value.reinterpret(domain))
assert V.OfAny(x.reinterpret(I8), {"x": None}) is None

## DOM-16 · pack gives a packed value's code in its representation, and unpack the packed value of a code

In [ ]:
Int2 = D.OfInteger.Data(2)
Signed = D.OfPacked.Data(State, Int2, (-2, 1))
assert V.OfAny(L("RUN", Coded).pack()) == D.Value(Bits2, b"\x01") and V.OfAny(L("IDLE", Signed).pack()) == D.Value(Int2, -2)
assert V.OfAny(L(b"\x01", Bits2).unpack(Coded)) == D.Value(Coded, "RUN") and V.OfAny(L(-2, Int2).unpack(Signed)) == D.Value(Signed, "IDLE")
assert V.OfAny(L("IDLE", Coded).pack().unpack(Coded)) == D.Value(Coded, "IDLE")
with raises(ValueError, match="no member of packed(enum(IDLE, RUN) as bits2) has code 3"):
    V.OfAny(L(b"\x03", Bits2).unpack(Coded))
with raises(TypeError, match="unpack expects bits2, got int"):
    V.OfAny(L(1).unpack(Coded))
with raises(TypeError, match="unpack needs a packed domain, got int8"):
    V.OfAny(L(1, I8).unpack(I8))
with raises(TypeError, match="pack expects a packed value, got enum(IDLE, RUN)"):
    V.OfAny(L("RUN", State).pack())
assert V.OfAny(x.pack(), {"x": None}) is None and V.OfAny(x.unpack(Coded), {"x": None}) is None